# API Abuse & Rate Limiting Analysis
This notebook sets up a local SQL database (using Python's built-in `sqlite3`) and performs data analytics entirely within Python. **No Docker or external database is required.**

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Create an in-memory database (or a local file like 'api_abuse.db')
conn = sqlite3.connect('api_abuse.db')
cursor = conn.cursor()
print("Database connected successfully!")

### 1. SQL Schema Setup
We will create the tables needed for clients, endpoints, and request logs.

In [ ]:
cursor.executescript('''
CREATE TABLE IF NOT EXISTS clients (
    client_id INTEGER PRIMARY KEY AUTOINCREMENT,
    client_name TEXT NOT NULL,
    plan_type TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS endpoints (
    endpoint_id INTEGER PRIMARY KEY AUTOINCREMENT,
    path TEXT NOT NULL,
    method TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS request_logs (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    client_id INTEGER,
    endpoint_id INTEGER,
    status_code INTEGER,
    request_time DATETIME DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY(client_id) REFERENCES clients(client_id),
    FOREIGN KEY(endpoint_id) REFERENCES endpoints(endpoint_id)
);
''')
conn.commit()
print("Tables created.")

### 2. Seed Data
Inserting sample data representing normal traffic and abuse behavior.

In [ ]:
cursor.executescript('''
INSERT OR IGNORE INTO clients (client_id, client_name, plan_type) VALUES 
    (1, 'Normal User', 'basic'),
    (2, 'Spam Bot', 'free'),
    (3, 'Scraper', 'premium');

INSERT OR IGNORE INTO endpoints (endpoint_id, path, method) VALUES 
    (1, '/api/v1/login', 'POST'),
    (2, '/api/v1/data', 'GET');
''')

# Simulating requests
import random
from datetime import datetime, timedelta

base_time = datetime.now() - timedelta(days=1)
logs = []

# Normal user: few requests
for _ in range(50):
    logs.append((1, 2, 200, base_time + timedelta(minutes=random.randint(1, 1440))))

# Spam Bot: Many failed login attempts
for _ in range(500):
    logs.append((2, 1, 401, base_time + timedelta(minutes=random.randint(1, 60))))

# Scraper: Massive amount of data reads
for _ in range(2000):
    logs.append((3, 2, 200, base_time + timedelta(minutes=random.randint(1, 1440))))

cursor.executemany(
    'INSERT INTO request_logs (client_id, endpoint_id, status_code, request_time) VALUES (?, ?, ?, ?)',
    logs
)
conn.commit()
print("Sample data inserted.")

### 3. Python & SQL Analytics
Now we analyze the data using Pandas to find abusers.

In [ ]:
query = '''
SELECT 
    c.client_name, 
    COUNT(r.log_id) as total_requests,
    SUM(CASE WHEN r.status_code >= 400 THEN 1 ELSE 0 END) as error_count
FROM request_logs r
JOIN clients c ON r.client_id = c.client_id
GROUP BY c.client_name
ORDER BY total_requests DESC;
'''
df = pd.read_sql_query(query, conn)
display(df)

df.plot(x='client_name', y='total_requests', kind='bar', title='Requests per Client', color='salmon')
plt.show()